In [1]:
import numpy as np 
import pandas as pd 
import MLM.angle_between as angle
import MLM.moire_lattice_vector_finder as mlf 
import MLM.structure_writer as sw
from matplotlib.path import Path

In [2]:
def select_atoms_in_polygon_path(df, polygon_path, tolerance):
    """
    Selects rows from a DataFrame where the 'x' and 'y' coordinates
    are inside or within a given tolerance of the specified polygon path.
    """
    xy_points = df[['x','y']].values  # Extract x, y coordinates
    # Add the radius parameter to include points near the boundary
    inside_mask = polygon_path.contains_points(xy_points, radius=tolerance)
    return df[inside_mask]

In [3]:
def write_vasp_optimized(filename, A1, A2, A3, atom_data, sort_order):
    # Buffer for all the lines to write
    element_type = " ".join(f"{key}" for key, _ in sort_order.items())
    element_count = " ".join(f"{len(sorted_df[sorted_df['type']==key])}" for key, _ in sort_order.items())
    lines = [
        "System Generated by MLM \n",
        "1.0\n",
        f"{A1[0]} {A1[1]} 0.0\n",
        f"{A2[0]} {A2[1]} 0.0\n",
        f"{A3[0]} {A3[1]} {A3[2]}\n",
        f"{element_type}\n",
        f"{element_count} \n",
        "Cartesian\n"
    ]
    atomic_positions = atom_data[['x', 'y', 'z']]
    # Write the buffer to the file
    with open(filename, 'w') as file:
        file.writelines(lines)
        # Use to_csv for efficient DataFrame writing
        atomic_positions.to_csv(file, sep=' ', index=False, header=False, mode='a')

In [12]:
file_1 = "../moire_structures/STO/STO_centered_unit_cell_tetra_monolayer.vasp"
file_2 = "../moire_structures/STO/STO_centered_unit_cell_tetra_monolayer.vasp"

In [13]:
lattice_vectors1, atom_type_arr1, dat1 = mlf.read_vasp(file_1)

lattice_vectors2, atom_type_arr2, dat2 = mlf.read_vasp(file_2)

In [17]:
df = pd.read_pickle('../moire_structures/STO/STO_bilayer_tetra_candidates.pkl')

In [18]:
df = df.sort_values(by='norm_A1',ascending=True).head(6)

In [19]:
df

,Theta,A1,A2,delvec,delang,norm_A1,norm_A2
60,36.87,"[-11.0446592602, 5.5223296301]","[-5.5223296301, -11.0446592602]",0.000022,0.0,12.348304,12.348304
37,22.62,"[-16.5669888903, 11.0446592602]","[-11.0446592602, -16.5669888903]",0.000047,0.0,19.911043,19.911043
47,28.07,"[-22.0893185204, -5.5223296301]","[-5.5223296301, 22.0893185204]",0.000988,0.0,22.769148,22.769148
27,16.26,"[-22.0893185204, 16.5669888903]","[-16.5669888903, -22.0893185204]",0.000099,0.0,27.611648,27.611648
71,43.60,"[-11.0446592602, 27.6116481505]","[-27.6116481505, -11.0446592602]",0.001463,0.0,29.738655,29.738655
32,18.92,"[-33.1339777806, -5.5223296301]","[-5.5223296301, 33.1339777806]",0.002723,0.0,33.591020,33.591020


In [21]:
import warnings
warnings.filterwarnings('ignore')


count = 1
for row in df.iterrows():
    A1 = np.array(row[1]['A1'])
    A2 = np.array(row[1]['A2'])
    norm = row[1]['norm_A1'] + row[1]['norm_A2']
    replicate = int((4*norm/np.linalg.norm(lattice_vectors1['a'])))
    new_lattice_vectors1, new_total_atoms1, replicated_atom1, replicated_type_arr1 = sw.replicate_atoms(a = lattice_vectors1['a'],
                                                                                             b = lattice_vectors1['b'],
                                                                                             c = lattice_vectors1['c'],
                                                                                             atom_data = dat1,
                                                                                             atom_type_arr = atom_type_arr1,
                                                                                             natoms = len(dat1),
                                                                                             na = replicate,
                                                                                             nb = replicate,
                                                                                             nc = 1)
    new_lattice_vectors2, new_total_atoms2, replicated_atom2, replicated_type_arr2 = sw.replicate_atoms(a = lattice_vectors2['a'],
                                                                                             b = lattice_vectors2['b'],
                                                                                             c = lattice_vectors2['c'],
                                                                                             atom_data = dat2,
                                                                                             atom_type_arr = atom_type_arr2,
                                                                                             natoms = len(dat2),
                                                                                             na = replicate,
                                                                                             nb = replicate,
                                                                                             nc = 1)

    # Create a DataFrame for positions
    positions1_df = pd.DataFrame(replicated_atom1, columns=['x', 'y', 'z'])

    # Add the atom types to the DataFrame
    positions1_df['type'] = replicated_type_arr1
    
    positions2_df = pd.DataFrame(replicated_atom2, columns=['x', 'y', 'z'])

    # Add the atom types to the DataFrame
    positions2_df['type'] = replicated_type_arr2 
    positions2_df['z'] = positions2_df['z'] + 3 + 8
    

    
    theta1 = row[1]['Theta']
    
    theta_rad = np.deg2rad(theta1)
    
    rotation_matrix = np.array([[np.cos(theta_rad), -np.sin(theta_rad), 0],
                            [np.sin(theta_rad), np.cos(theta_rad), 0],
                            [0, 0, 1]])
    
    middle_layer_positions = positions2_df[['x', 'y', 'z']].values  # Extract x, y, z positions
    rotated_positions = middle_layer_positions @ rotation_matrix.T  # Apply rotation

    # Create a new DataFrame for the rotated positions
    rotated_middle_layer = pd.DataFrame(rotated_positions, columns=['x', 'y', 'z'])

    # Add the atom types back to the rotated DataFrame
    rotated_middle_layer['type'] = positions2_df['type'].values
    
    
    
    # Concatenate DataFrames vertically
    concat_vertical = pd.concat([positions1_df, rotated_middle_layer], ignore_index=True)
    

    A3 = np.array([0, 0, 50.0])
    
    vertex = [ [0.0,0.0], [A1[0], A1[1]],  [(A1+A2)[0],(A1+A2)[1]], [A2[0],A2[1]]]
    
    polygon_points = vertex
    
    polygon_path = Path(polygon_points)
    
    selected_atoms_df = select_atoms_in_polygon_path(concat_vertical, polygon_path, tolerance=-0.01)
    
    unique_types = selected_atoms_df['type'].unique()
    
    sort_order = {value: index for index, value in enumerate(unique_types)}
    
    selected_atoms_df['sort_key'] = selected_atoms_df['type'].map(sort_order)
    
    sorted_df = selected_atoms_df.sort_values(by='sort_key').drop(columns='sort_key').copy()
    
    sorted_df = sorted_df.round(5)
    
    write_vasp_optimized(f"../moire_structures/STO/STO_moire_tetra/sto_tetra_{np.round(theta1,2)}.vasp",A1,A2,A3, sorted_df, sort_order)
    print(f"Done with {count} | {np.round(theta1,2)}")
    count = count + 1
    
    
    


Done with 1 | 36.87
Done with 2 | 22.62
Done with 3 | 28.07
Done with 4 | 16.26
Done with 5 | 43.6
Done with 6 | 18.92
